# GreenSpace CNN - Core Pipeline v1

This is the PyTorch/TorchGeo handoff notebook for demonstrating the GreenSpace pipeline from Google Drive authentication through a trained-model evaluation. It calls reusable project functions instead of copying implementation logic from the historical notebooks.

## Current scaffold status

- Ready: project setup and explicit paths
- Ready: existing Google Drive OAuth authentication
- Ready: rated-image download as a separate terminal command
- Ready: reusable preprocessing function and separate terminal command
- Ready: survey cleaning, inclusion filtering, and rater aggregation
- Ready: deterministic 50-image demonstration selection and 60/20/20 split
- Ready: current TorchGeo backbone/model configuration
- Ready: existing one-warm-up plus one-fine-tuning test orchestration
- Ready: resumable PyTorch training terminal command
- Ready: existing evaluation and validation-threshold orchestration
- Ready: packaged evaluation terminal command that saves calibrated thresholds

> The 50-image, two-epoch run is functional smoke evidence only. Its metrics are not model-performance evidence and must not be compared with a full training run.

## 1. Setup and explicit inputs

Set `GREENSPACE_RAW_SURVEY` to the raw survey CSV before running the preprocessing cells. The expected headers and image layout are the same as the existing Notebook 02 workflow.


In [ ]:
from pathlib import Path
import os
import random
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

raw_survey_env = os.getenv('GREENSPACE_RAW_SURVEY')
RAW_SURVEY_PATH = Path(raw_survey_env).expanduser() if raw_survey_env else None
IMAGE_CACHE_DIR = PROJECT_ROOT / 'data' / 'cache' / 'images'
DEMO_DATA_ROOT = PROJECT_ROOT / 'data' / 'core_pipeline_demo'
DEMO_SPLIT_DIR = DEMO_DATA_ROOT / 'processed' / 'splits'

PREPROCESSING_SEED = 123
SPLIT_SEED = 123
SPLIT_SEED_2 = 456
PYTORCH_SEED = 37
DEMO_SAMPLE_SIZE = 50

random.seed(PREPROCESSING_SEED)
np.random.seed(PREPROCESSING_SEED)

RUN_GOOGLE_AUTH = False
RUN_MODEL_BUILD = False
RUN_DEMO_TRAINING = False
RUN_DEMO_EVALUATION = False

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_SURVEY_PATH:', RAW_SURVEY_PATH or 'SET GREENSPACE_RAW_SURVEY')
print('IMAGE_CACHE_DIR:', IMAGE_CACHE_DIR)
print('DEMO_SPLIT_DIR:', DEMO_SPLIT_DIR)
print('Seeds:', {'preprocessing': PREPROCESSING_SEED, 'split_1': SPLIT_SEED, 'split_2': SPLIT_SEED_2, 'pytorch': PYTORCH_SEED})


## 2. Google Drive authentication

This uses the existing PyDrive2 OAuth flow in `src/drive_utils.py`. Google Cloud Console is used to create the OAuth desktop credentials; the images themselves are read from Google Drive. See `instruction_docs/google_drive_auth.md`.

Set `RUN_GOOGLE_AUTH = True` in the setup cell when you want the browser authentication flow to run.


In [ ]:
from src.drive_utils import get_drive

drive = None
if RUN_GOOGLE_AUTH:
    drive = get_drive(use_local_server=True)
    print('Google Drive authentication succeeded.')
else:
    print('Authentication skipped. Set RUN_GOOGLE_AUTH=True when Drive access is needed.')


## 3. Rated-image download - terminal prerequisite

Run the packaged downloader from the repository root before continuing with the cached-image sections of this notebook:

```bash
python scripts/download_drive_images.py --survey-csv data/raw/0614_survey_response.csv --filelist-csv data/filelist_0103.csv --cache-dir data/cache/images
```

Replace both CSV paths for the current run. This notebook intentionally does not invoke the downloader; the following preprocessing and split cells expect the required rated images to already exist under `data/cache/images/`. See `instruction_docs/google_drive_auth.md` for browser authentication, manifest-only inspection, and a 50-image test command.


## 4. Run the packaged preprocessing pipeline

The concise terminal entry point calls the same function demonstrated in the next cell. For a full cached dataset, run this from the repository root:

```bash
python scripts/preprocess.py --survey-csv data/raw/0614_survey_response.csv --filelist-csv data/interim/filelist_0418_215415_with_drive_fileid.csv
```

Replace both paths for the current run. The function call below uses the same cleaning, `include_tile == yes` filtering, label aggregation, and seeded 60/20/20 splitting behavior, but explicitly selects 50 cached images for this demonstration.


In [ ]:
from src.preprocessing import run_preprocessing_pipeline

if RAW_SURVEY_PATH is None:
    raise ValueError('Set GREENSPACE_RAW_SURVEY to the raw survey CSV before running this cell.')
if not RAW_SURVEY_PATH.is_file():
    raise FileNotFoundError(f'Missing raw survey CSV: {RAW_SURVEY_PATH}')

filelist_env = os.getenv('GREENSPACE_FILELIST_WITH_IDS')
FILELIST_WITH_IDS_PATH = Path(filelist_env).expanduser() if filelist_env else None
if FILELIST_WITH_IDS_PATH is not None and not FILELIST_WITH_IDS_PATH.is_file():
    raise FileNotFoundError(f'Missing filelist with Drive IDs: {FILELIST_WITH_IDS_PATH}')

preprocessing_result = run_preprocessing_pipeline(
    survey_path=RAW_SURVEY_PATH,
    image_dir=IMAGE_CACHE_DIR,
    run_tag='CORE_demo_v1',
    interim_dir=DEMO_DATA_ROOT / 'interim',
    processed_dir=DEMO_DATA_ROOT / 'processed',
    split_dir=DEMO_SPLIT_DIR,
    filelist_path=FILELIST_WITH_IDS_PATH,
    split_seed=SPLIT_SEED,
    split_seed_2=SPLIT_SEED_2,
    sample_size=DEMO_SAMPLE_SIZE,
    sample_seed=PYTORCH_SEED,
    require_all_images=True,
)

labels_soft = pd.read_csv(preprocessing_result['paths']['labels_soft'])
labels_hard = pd.read_csv(preprocessing_result['paths']['labels_hard'])
print('Inclusion summary:', preprocessing_result['inclusion_summary'])
print('Selection summary:', preprocessing_result['selection_summary'])
print('Split summary:', preprocessing_result['split_summary'])
display(labels_soft.head())
display(labels_hard.head())


## 5. Inspect the 50-image demonstration split

The preprocessing function selected 50 cached, labeled images deterministically with PyTorch seed `37`, then applied the existing Notebook 02 split seeds and ratios. This produces 30/10/10 manifests.


In [ ]:
demo_split_summary = preprocessing_result['split_summary']
demo_split_paths = {
    split: preprocessing_result['paths'][split]
    for split in ('train', 'val', 'test')
}
expected_sizes = {'train': 30, 'val': 10, 'test': 10}
actual_sizes = {split: len(pd.read_csv(path)) for split, path in demo_split_paths.items()}
if actual_sizes != expected_sizes:
    raise ValueError(f'Unexpected demonstration split sizes: {actual_sizes}')
print('Demo split summary:', demo_split_summary)
for split, path in demo_split_paths.items():
    print(f'{split}: {path}')


## 6. Inspect the current PyTorch/TorchGeo model

This uses the current canonical configuration: TorchGeo Swin V2 Base with NAIP RGB Satlas weights, seven active binary outputs, shade classification, and bounded continuous score/vegetation heads. Building the pretrained backbone may download its weights when they are not already cached.


In [ ]:
from src_torch.config import TORCH_DATA_CONFIG, TORCH_MODEL_CONFIG, TORCH_TRAINING_CONFIG
from src_torch.models import build_torchgeo_model
from src_torch.training import trainable_parameter_summary

print('Data configuration:', TORCH_DATA_CONFIG)
print('Model configuration:', TORCH_MODEL_CONFIG)
print('Training configuration:', TORCH_TRAINING_CONFIG)

demo_model = None
if RUN_MODEL_BUILD:
    demo_model = build_torchgeo_model()
    print('Model metadata:', demo_model.metadata())
    print('Parameter summary:', trainable_parameter_summary(demo_model))
else:
    print('Model build skipped. Set RUN_MODEL_BUILD=True to instantiate the configured backbone and heads.')


## 7. Run one warm-up epoch and one fine-tuning epoch

This calls the same resumable PyTorch trainer used by the terminal command with `test_run_mode=True`. The first epoch freezes the backbone and trains the GreenSpace heads. The second epoch unfreezes the configured backbone and fine-tunes end to end.

Set `RUN_DEMO_TRAINING = True` only after the 30/10/10 demo manifests exist.

Equivalent terminal command:

```bash
python scripts/train_torch.py --mode smoke --data-root data/core_pipeline_demo --device auto
```

After an interruption, repeat that command with `--resume models/runs/<run-tag>/last_<run-tag>.pt`. Resume starts after the last completed epoch; an interrupted partial epoch is repeated.


In [ ]:
training_result = None
if RUN_DEMO_TRAINING:
    if not all(path.is_file() for path in demo_split_paths.values()):
        raise FileNotFoundError('Create the demonstration split manifests before training.')
    from src_torch.training import run_persistent_warmup_finetune

    training_result = run_persistent_warmup_finetune(
        training_config={
            'seed': PYTORCH_SEED,
            'test_run_mode': True,
            'device': 'auto',
            'max_train_batches': None,
            'max_val_batches': None,
        },
        split_dir=DEMO_SPLIT_DIR,
        image_root=IMAGE_CACHE_DIR,
    )
    display(training_result)
else:
    print('Demo training skipped. Set RUN_DEMO_TRAINING=True to run the existing 1+1 test schedule.')


## 8. Evaluate the trained demonstration model

This reuses the existing PyTorch evaluation functions: predict train/validation/test, calculate monitoring metrics, tune binary thresholds on validation, and save the evaluation artifacts beside the demo run.

Equivalent terminal command (now packaged):

```bash
python scripts/evaluate_torch.py --checkpoint models/runs/<demo-run>/best_mcmae_<demo-run>.pt
```

Omit `--checkpoint` to auto-select the newest run's best-MC-MAE checkpoint. The command resolves splits and images from the repository `data/` directory by default; point it elsewhere with `GREENSPACE_DATA_ROOT` (see README section 3a). It writes `thresholds_<variant>.csv` beside the checkpoint, plus the loss-monitor and per-split report CSVs.

In [ ]:
evaluation_paths = None
if RUN_DEMO_EVALUATION:
    if training_result is None:
        raise ValueError('Run the demonstration training cell before evaluation.')

    from src_torch.data import load_split_dfs, resolve_split_schema
    from src_torch.evaluation import (
        evaluate_all_splits,
        evaluate_loss_monitoring,
        infer_run_tag_and_variant,
        load_torch_checkpoint_model,
        predict_split,
        save_evaluation_outputs,
        tune_val_thresholds,
    )
    from src_torch.training import resolve_device

    checkpoint_path = Path(training_result['best_mc_mae_path'])
    device = resolve_device('auto')
    trained_model, _, _ = load_torch_checkpoint_model(checkpoint_path, device=device)
    run_tag, variant = infer_run_tag_and_variant(checkpoint_path)
    split_dfs = load_split_dfs()
    schema = resolve_split_schema(split_dfs['train'])

    predictions_by_split = {
        split: predict_split(trained_model, split, device=device, batch_size=int(TORCH_DATA_CONFIG['batch_size']))
        for split in ('train', 'val', 'test')
    }
    loss_monitor = evaluate_loss_monitoring(predictions_by_split, schema.binary_cols)
    thresholds, threshold_map = tune_val_thresholds(
        predictions_by_split['val'],
        schema.bin_names,
        schema.binary_cols,
    )
    overall, per_label = evaluate_all_splits(
        predictions_by_split,
        schema.binary_cols,
        threshold_map,
    )
    evaluation_paths = save_evaluation_outputs(
        run_tag=run_tag,
        variant=variant,
        loss_monitor_df=loss_monitor,
        thresholds_df=thresholds,
        overall_df=overall,
        per_label_df=per_label,
    )
    display(overall.round(4))
    display(per_label.round(4))
    print('Saved evaluation artifacts:', evaluation_paths)
else:
    print('Demo evaluation skipped. Set RUN_DEMO_EVALUATION=True after demo training succeeds.')

## 9. Production workflow after the demonstration

The handoff now exposes Drive download, preprocessing, resumable training, and evaluation as separate terminal commands. Prediction on new images still runs through `notebooks/05_pyTorch_prediction_demo.ipynb`; packaging it as a command is the remaining step. These commands support the full uncapped dataset derived from the approximately 10,000 raw survey rows.

`scripts/evaluate_torch.py` already produces the calibrated `thresholds_<variant>.csv` artifact beside the checkpoint, which is the threshold half of a complete prediction bundle.